## Preperations (execute cells, no need to change anything)

In [ ]:
import numpy as np
import xarray as xr
import dask.bag as db
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.optimize import brentq
from IPython.display import display, clear_output

In [ ]:
def compute_signal_ida(params, g0_values, Kd, h0, d0):
        I0, Kg, Id, Ihd = params
        Signal_values = []
        for g0 in g0_values:
            try:
                def equation_h(h):
                    denom_Kd = 1 + Kd * h
                    denom_Kg = 1 + Kg * h
                    h_d = (Kd * h * d0) / denom_Kd
                    h_g = (Kg * h * g0) / denom_Kg
                    return h + h_d + h_g - h0

                h_sol = brentq(equation_h, 1e-20, h0, xtol=1e-14, maxiter=1000)
                denom_Kd = 1 + Kd * h_sol
                d_free = d0 / denom_Kd
                h_d = Kd * h_sol * d_free
                h = h0 - h_d
                Signal = I0 + Id * d_free + Ihd * h_d
                Signal_values.append(Signal)
            except Exception:
                Signal_values.append(np.nan)
        return np.array(Signal_values)

In [ ]:
concentration_vector = np.linspace(0, 1, 11)
def synthesize_data(parameter_ranges, N, Kga_sampling, Kd_sampling):
    
    def sample_parameters(N):
        I0 = np.random.uniform(*parameter_ranges['I0'], size=N)
        Ihd = np.exp(np.random.uniform(*np.log(parameter_ranges['Ihd']), size=N))
        Id = np.exp(np.random.uniform(*np.log(parameter_ranges['Id']), size=N))
        h0 = np.random.uniform(*parameter_ranges['h0'], size=N)
        d0 = np.random.uniform(*parameter_ranges['d0'], size=N)
        Kd = Kd_sampling()
        Kga = Kga_sampling()
        
        
        return [I0, Kga, Id, Ihd, Kd, h0, d0]

    # for Dask parallelization
    def compute_signal_dask(params):
        I0, Kga, Id, Ihd, Kd, h0, d0 = params
        
        signal = compute_signal_ida([I0, Kga, Id, Ihd], concentration_vector, Kd, h0, d0)
        parameters = np.asarray([I0, Id, Ihd, h0, d0, Kga, Kd])
        return parameters, signal

    # dask parallelization
    parameters_bag = db.from_sequence(zip(*sample_parameters(N)))
    results_bag = parameters_bag.map(compute_signal_dask)
    results = results_bag.compute()

    params_dask, signals = zip(*results)
    params_dask = np.asarray(params_dask)
    signals = np.asarray(signals)

    ds = xr.Dataset(
        {
            "signal": (("index", "nPoints"), signals),
            "I0": ("index", params_dask[:, 0]),
            "Id": ("index", params_dask[:, 1]),
            "Ihd": ("index", params_dask[:, 2]),
            "h0": ("index", params_dask[:, 3]),
            "d0": ("index", params_dask[:, 4]),
            "Kga": ("index", params_dask[:, 5]),
            "Kd": ("index", params_dask[:, 6]),
        },
        coords={
            "index": range(N),
            "nPoints": concentration_vector,
        }
    )
    
    return ds

In [ ]:
def plot_Kd_Kga_distributions(Kga, Kd, bins=100):
    
    logKga = np.log10(Kga)
    logKd = np.log10(Kd)

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Plot Kd & Kga distributions together
    ax1 = axes[0]
    ax1.minorticks_on()
    ax1.hist(Kd, bins=bins, color='blue', alpha=0.5, label='Kd')
    ax1.hist(Kga, bins=bins, color='orange', alpha=0.5, label='Kga')
    ax1.set_xlabel('Value (Log Scale)')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Kd and Kga Distributions')
    ax1.set_xscale('log')
    ax1.legend()
    ax1.grid(which='both', linestyle='--', linewidth=0.5)

    # Plot logKd & logKga distributions together
    ax2 = axes[1]
    ax2.hist(logKga, bins=bins, color='orange', alpha=0.5, label='log10(Kga)')
    ax2.hist(logKd, bins=bins, color='blue', alpha=0.5, label='log10(Kd)')
    ax2.set_xlabel('log10 Value')
    ax2.set_ylabel('Frequency')
    ax2.set_title('log10(Kd) and log10(Kga) Distributions')
    ax2.legend()
    ax2.minorticks_on()
    ax2.grid(which='both', linestyle='--', linewidth=0.5)

    plt.tight_layout()
    plt.show()

In [ ]:
%matplotlib inline
def plot_samples_from_signals(signals, labels=None, normalize=False):
    
    if normalize:
        # normalize signals
        signals = signals / signals.max(axis=1, keepdims=True)
    
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.set_xlabel('Concentration')
    ax.set_ylabel('Signal')
    ax.grid(which='both', linestyle='--', linewidth=0.5)
    title = ax.set_title("Signals")
    
    lines = [ax.plot([])[0] for i in range(10)]
    plt.tight_layout()
    plt.close(fig)

    slider = widgets.IntSlider(
        min=0,
        max=max(0, signals.shape[0] - len(lines)),
        step=len(lines),
        value=0,
        description='Batch:',
        continuous_update=False
    )
    out = widgets.Output()

    def update(batch_start):
        batch_end = batch_start + len(lines)
        batch = signals[batch_start:batch_end]
        y_min, y_max = batch.min(), batch.max()
        margin = (y_max - y_min) * 0.05

        ax.set_xlim(concentration_vector.min(), concentration_vector.max())
        ax.set_ylim(y_min - margin, y_max + margin)

        with out:
            clear_output(wait=True)
            for i, line in enumerate(lines):
                idx = batch_start + i
                if idx < signals.shape[0]:
                    line.set_data(concentration_vector, signals[idx])
                    if labels is not None:
                        line.set_label(f"Id: {labels[idx][0]}, Ihd: {labels[idx][1]}, h0: {labels[idx][2]}, d0: {labels[idx][3]}, Kga: {labels[idx][4]}, Kd: {labels[idx][5]}")
                    line.set_visible(True)
                else:
                    line.set_visible(False)
            title.set_text(f"Signals {batch_start} to {min(batch_end-1, signals.shape[0]-1)}")
            # fig.tight_layout()
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
            display(fig)

    slider.observe(lambda change: update(change['new']), names='value')
    display(slider, out)
    update(0)

In [ ]:
N = 10000
save = False

params = {
    'g0' : 1.0,
    'h0': (0.3, 1.0),
    'd0': (0.3, 1.0),
    'Kd': (0.1, 10000),
    # 'Kg': (1, 1000), 
    'I0': (0.0, 0),
    'Id': (1, 1000),
    'Ihd': (1, 1000),
}

## Frank's Suggestion

### Kd vs Kga Distribution

In [ ]:
# Frank's suggestion
factor_sampling = lambda : np.exp(np.random.uniform(*np.log((0.1, 1000)), size=N))
Kd_sampling = lambda : np.exp(np.random.uniform(*np.log(params['Kd']), size=N))
Kga_sampling = lambda : Kd_sampling() * factor_sampling()

ds_factor = synthesize_data(params, N, Kd_sampling=Kd_sampling, Kga_sampling=Kga_sampling)
plot_Kd_Kga_distributions(ds_factor.Kga.values, ds_factor.Kd.values, bins=100)

### Plot All Data

In [ ]:
signals = ds_factor['signal'].values

Id = ds_factor['Id'].values
Ihd = ds_factor['Ihd'].values
h0 = ds_factor['h0'].values
d0 = ds_factor['d0'].values

Kga = ds_factor['Kga'].values
Kd = ds_factor['Kd'].values

labels = np.column_stack([Id, Ihd, h0, d0, Kga, Kd])
labels = np.round(labels, 3)

plot_samples_from_signals(signals, labels, normalize=True)

### Filter Out Data with Kd $\cdot$ Kga < 1

In [ ]:
# filter out signals where Kd * Kga < 1
mask = (Kd * Kga) > 1
signals_filtered = signals[mask]
labels_filtered = labels[mask]

# Sanity check
print(f"Original signals shape: {signals.shape}")
print(f"Filtered signals shape: {signals_filtered.shape}")

plot_samples_from_signals(signals_filtered, labels_filtered, normalize=True)

## From Stephan's Paper

### Kd vs Kga Distribution

In [ ]:
# Kd
Kd_range = (0.1, 10000)
Kd_sampling = lambda : np.exp(np.random.uniform(*np.log(Kd_range), size=N))

# Kga
Kga_range = (0.001, 100)
Kga_sampling = lambda : np.exp(np.random.uniform(*np.log(Kga_range), size=N))

ds_ = synthesize_data(params, N, Kd_sampling=Kd_sampling, Kga_sampling=Kga_sampling)
plot_Kd_Kga_distributions(ds_.Kga.values, ds_.Kd.values, bins=100)

### Plot All Data

In [ ]:
signals = ds_['signal'].values

Id = ds_['Id'].values
Ihd = ds_['Ihd'].values
h0 = ds_['h0'].values
d0 = ds_['d0'].values

Kga = ds_['Kga'].values
Kd = ds_['Kd'].values

labels = np.column_stack([Id, Ihd, h0, d0, Kga, Kd])
labels = np.round(labels, 3)

plot_samples_from_signals(signals, labels, normalize=True)

### Filter Out Data with Kd $\cdot$ Kga < 1

In [ ]:
# filter out signals where Kd * Kga < 1
mask = (Kd * Kga) > 1
signals_filtered = signals[mask]
labels_filtered = labels[mask]

# Sanity check
print(f"Original signals shape: {signals.shape}")
print(f"Filtered signals shape: {signals_filtered.shape}")

plot_samples_from_signals(signals_filtered, labels_filtered, normalize=True)